<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/AdvancedAgent/ipynb/2.1_travel_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-openai langchain-mcp-adapters \
    tavily-python mcp

In [1]:
import warnings
warnings.filterwarnings("ignore")
import dotenv

_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

In [2]:
from langchain_mcp_adapters import client

client = client.MultiServerMCPClient(connections={
    "travel_server": {
        "transport": "streamable_http",
        "url": "https://mcp.kiwi.com"
    }
})

tools = await client.get_tools()

In [3]:
import os
import time
from langchain import chat_models, agents, messages
from langgraph.checkpoint import memory

llm = chat_models.init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

agent = agents.create_agent(
    model=llm,
    tools=tools,
    checkpointer=memory.InMemorySaver(),
    system_prompt="""
        You are a travel agent.
        Always provide both departureDate and returnDate in dd/mm/yyyy format
        when calling tools.
        If user asks one-way, set returnDate to departureDate.
        No follow up questions.
    """
)

config = {"configurable": {"thread_id": "1"}}
start_time = time.time()
response = await agent.ainvoke(
    {"messages": [messages.HumanMessage(content="""
        Get me a direct flight from Kuala Lumpur to Tokyo on 31/03/2026
        and return on 11/04/2026.
    """)]},
    config=config
)
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 22.41s
================================ Human Message =================================


        Get me a direct flight from Kuala Lumpur to Tokyo on 31/03/2026
        and return on 11/04/2026.
    
================================== Ai Message ==================================
Tool Calls:
  search-flight (call_ap1naa6s)
 Call ID: call_ap1naa6s
  Args:
    cabinClass: M
    curr: USD
    departureDate: 31/03/2026
    departureDateFlexRange: 0
    flyFrom: Kuala Lumpur
    flyTo: Tokyo
    locale: en
    passengers: {'adults': 1}
    returnDate: 11/04/2026
    returnDateFlexRange: 0
    sort: price
================================= Tool Message =================================
Name: search-flight

[{'type': 'text', 'text': '[\n  {\n    "flyFrom": "KUL",\n    "flyTo": "HND",\n    "cityFrom": "Kuala Lumpur",\n    "cityTo": "Tokyo",\n    "departure": {\n      "utc": "2026-03-31T04:20:00.000Z",\n      "local": "2026-03-31T12:20:00.000"\n    },\n    "arrival": {\n      "utc":